# Семинар 2. Исследование методов линейной регрессии

#### Критерий оценивания:
#### Пункт 14: максимум 0,5 балла
#### Пункт 15: максимум 0,5 балла

#### Итого за работу: максимум 1 балл
#### P.S. пункты 14 и 15 без пунктов 1-13 не засчитываются, ответы на вопросы из ИИ не засчитываются

1. Загрузите датасет и выведите на экран первые несколько строк

In [30]:
import pandas as pd

# Загрузка датасета
data = pd.read_csv(r"C:\Users\Lecoo\Downloads\auto_dataset.csv")

# Вывод первых строк датасета
data.head()

,brand,model,vehicleType,gearbox,fuelType,notRepairedDamage,powerPS,kilometer,autoAgeMonths,price
0,volkswagen,golf,kleinwagen,manuell,benzin,nein,75,150000,177,1500
1,skoda,fabia,kleinwagen,manuell,diesel,nein,69,90000,93,3600
2,bmw,3er,limousine,manuell,benzin,ja,102,150000,246,650
3,peugeot,2_reihe,cabrio,manuell,benzin,nein,109,150000,140,2200
4,mazda,3_reihe,limousine,manuell,benzin,nein,105,150000,136,2000


2. Разбейте выборку на признаки и ответы. Закодируйте категориальные признаки.

In [31]:
# Категориальные и числовые признаки
cat_cols = ['brand', 'model', 'vehicleType', 'gearbox', 'fuelType', 'notRepairedDamage']
num_cols = ['powerPS', 'kilometer', 'autoAgeMonths']
target_col = 'price'

# Перемешиваем данные (чтобы не было сортировки по марке/модели)
data = data.sample(frac=1, random_state=42).reset_index(drop=True)

# Разбиение 8:1:1
n = len(data)
train_end = int(0.8 * n)
val_end = int(0.9 * n)

train_data = data.iloc[:train_end].reset_index(drop=True)
val_data = data.iloc[train_end:val_end].reset_index(drop=True)
test_data = data.iloc[val_end:].reset_index(drop=True)

# X и y
X_train = train_data[num_cols + cat_cols]
y_train = train_data[target_col]

X_val = val_data[num_cols + cat_cols]
y_val = val_data[target_col]

X_test = test_data[num_cols + cat_cols]
y_test = test_data[target_col]

print("Размеры выборок:")
print(f"train: {X_train.shape}, val: {X_val.shape}, test: {X_test.shape}")

Размеры выборок:
train: (800, 9), val: (100, 9), test: (100, 9)


3. Разбейте датасет на train val test в отношении 8:1:1

In [32]:
# One-Hot Encoding
X_train_enc = pd.get_dummies(X_train, columns=cat_cols, drop_first=True)
X_val_enc = pd.get_dummies(X_val, columns=cat_cols, drop_first=True)
X_test_enc = pd.get_dummies(X_test, columns=cat_cols, drop_first=True)

# Выравниваем столбцы val и test по train
X_val_enc = X_val_enc.reindex(columns=X_train_enc.columns, fill_value=0)
X_test_enc = X_test_enc.reindex(columns=X_train_enc.columns, fill_value=0)

print("Размеры после кодирования:")
print(f"X_train_enc: {X_train_enc.shape}")
print(f"X_val_enc:   {X_val_enc.shape}")
print(f"X_test_enc:  {X_test_enc.shape}")

# Масштабирование числовых признаков
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
scaler.fit(X_train_enc[num_cols])

X_train_enc[num_cols] = scaler.transform(X_train_enc[num_cols])
X_val_enc[num_cols] = scaler.transform(X_val_enc[num_cols])
X_test_enc[num_cols] = scaler.transform(X_test_enc[num_cols])

print("Числовые признаки стандартизированы.")
display(X_train_enc.head())

# В numpy + добавление bias
import numpy as np

X_train_np = X_train_enc.values.astype(float)
X_val_np = X_val_enc.values.astype(float)
X_test_np = X_test_enc.values.astype(float)

y_train_np = y_train.values.astype(float).reshape(-1, 1)
y_val_np = y_val.values.astype(float).reshape(-1, 1)
y_test_np = y_test.values.astype(float).reshape(-1, 1)

# Добавляем столбец единиц (bias)
X_train_np = np.hstack([np.ones((X_train_np.shape[0], 1)), X_train_np])
X_val_np = np.hstack([np.ones((X_val_np.shape[0], 1)), X_val_np])
X_test_np = np.hstack([np.ones((X_test_np.shape[0], 1)), X_test_np])

print(f"X_train_np: {X_train_np.shape}")
print(f"X_val_np:   {X_val_np.shape}")
print(f"X_test_np:  {X_test_np.shape}")
print(f"y_train_np: {y_train_np.shape}")
def mse(X, y, w):
    """Loss = (1/l) * ||Xw - y||^2"""
    l = X.shape[0]
    return float((1.0 / l) * np.sum((X @ w - y) ** 2))


def r2_score(y_true, y_pred):
    """R^2 = 1 - SS_res / SS_tot"""
    ss_res = np.sum((y_true - y_pred) ** 2)
    ss_tot = np.sum((y_true - y_true.mean()) ** 2)
    return 1.0 - ss_res / ss_tot


def evaluate(X, y, w):
    """Loss и R^2 для модели с весами w"""
    return mse(X, y, w), r2_score(y, X @ w)

Размеры после кодирования:
X_train_enc: (800, 196)
X_val_enc:   (100, 196)
X_test_enc:  (100, 196)
Числовые признаки стандартизированы.


,powerPS,kilometer,autoAgeMonths,brand_audi,brand_bmw,brand_chevrolet,brand_chrysler,brand_citroen,brand_dacia,brand_daewoo,...,vehicleType_kleinwagen,vehicleType_kombi,vehicleType_limousine,vehicleType_suv,gearbox_manuell,fuelType_benzin,fuelType_diesel,fuelType_hybrid,fuelType_lpg,notRepairedDamage_nein
0,-0.5364,0.6682,-0.3781,False,False,False,False,False,False,False,...,False,False,False,False,True,False,False,False,True,True
1,0.2524,0.0390,-0.3646,False,False,False,False,False,False,False,...,False,True,False,False,False,False,True,False,False,True
2,-0.3063,0.6682,0.7389,False,False,False,False,False,False,False,...,False,False,True,False,True,True,False,False,False,True
3,-0.7336,-0.8419,5.30,False,False,False,False,False,False,False,...,False,False,True,False,False,True,False,False,False,True
4,-0.5035,0.6682,-0.4319,False,False,False,False,False,False,False,...,False,False,True,False,False,False,True,False,False,True


X_train_np: (800, 197)
X_val_np:   (100, 197)
X_test_np:  (100, 197)
y_train_np: (800, 1)


4. Исследуйте VGD с постоянным шагом n:

переберите n в логарифмической сетке от 10^-5 до 1

для каждого n:

* обучите VGD на train
* найдите и запомните R^2_train и Loss_train
* протестируйте VGD на val, запомните Loss_val

найдите наилучший n по минимальному Loss_val, запомните его Loss_train, R^2_train, Loss_val;

протестируйте VGD c лучшим n на test, запомните Loss_test, R^2_test, число итераций на test.

In [33]:
def vanilla_gd_const(X, y, n_step, n_iters=5000):
    """
    VanillaGD с ПОСТОЯННЫМ шагом.

    X, y     : train-данные
    n_step   : постоянный шаг η_k = n_step
    n_iters  : число итераций
    """
    l, d = X.shape
    w = np.zeros((d, 1))
    loss_hist = []

    for k in range(n_iters):
        grad = (2.0 / l) * X.T @ (X @ w - y)
        w = w - n_step * grad              # <-- постоянный шаг
        loss_hist.append(mse(X, y, w))

    return w, loss_hist


# Перебор постоянного шага
n_steps = [1e-5, 1e-4, 1e-3, 1e-2, 1e-1, 1.0]
results_vgd_const = []

for n_step in n_steps:
    w, loss_hist = vanilla_gd_const(X_train_np, y_train_np, n_step, n_iters=5000)

    loss_train = mse(X_train_np, y_train_np, w)
    r2_train = r2_score(y_train_np, X_train_np @ w)
    loss_val = mse(X_val_np, y_val_np, w)
    r2_val = r2_score(y_val_np, X_val_np @ w)

    results_vgd_const.append({
        'n': n_step,
        'loss_train': loss_train,
        'r2_train': r2_train,
        'loss_val': loss_val,
        'r2_val': r2_val,
        'n_iters': len(loss_hist),
        'w': w,
        'loss_hist': loss_hist,
    })
    print(f"n = {n_step:.0e} | Loss_train = {loss_train:12.2f} | R²_train = {r2_train: .4f} | "
          f"Loss_val = {loss_val:12.2f} | R²_val = {r2_val: .4f}")

# Лучший шаг по Loss_val
best_vgd_const = min(results_vgd_const, key=lambda r: r['loss_val'])
print(f"\nЛУЧШИЙ n (const) для VanillaGD: {best_vgd_const['n']:.0e}")
print(f"Loss_val = {best_vgd_const['loss_val']:.2f}, R²_val = {best_vgd_const['r2_val']:.4f}")

# Тест на лучшем n
w_best = best_vgd_const['w']
loss_test_vgd_const = mse(X_test_np, y_test_np, w_best)
r2_test_vgd_const = r2_score(y_test_np, X_test_np @ w_best)
print(f"Loss_test = {loss_test_vgd_const:.2f}, R²_test = {r2_test_vgd_const:.4f}")

n = 1e-05 | Loss_train =  72042252.38 | R²_train = -0.3380 | Loss_val =  94284815.25 | R²_val = -0.2975
n = 1e-04 | Loss_train =  22475280.40 | R²_train =  0.5826 | Loss_val =  34992807.31 | R²_val =  0.5184
n = 1e-03 | Loss_train =  17199534.86 | R²_train =  0.6806 | Loss_val =  23769984.20 | R²_val =  0.6729
n = 1e-02 | Loss_train =  14416325.62 | R²_train =  0.7322 | Loss_val =  21816072.34 | R²_val =  0.6998
n = 1e-01 | Loss_train =  12241289.69 | R²_train =  0.7726 | Loss_val =  21837821.85 | R²_val =  0.6995


C:\Users\Lecoo\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\numpy\_core\fromnumeric.py:83: RuntimeWarning: overflow encountered in reduce
  return ufunc.reduce(obj, axis, dtype, out, **passkwargs)
C:\Users\Lecoo\AppData\Local\Temp\ipykernel_6600\3981481072.py:51: RuntimeWarning: overflow encountered in square
  return float((1.0 / l) * np.sum((X @ w - y) ** 2))
C:\Users\Lecoo\AppData\Local\Temp\ipykernel_6600\3958950590.py:14: RuntimeWarning: overflow encountered in matmul
  grad = (2.0 / l) * X.T @ (X @ w - y)
C:\Users\Lecoo\AppData\Local\Temp\ipykernel_6600\3981481072.py:51: RuntimeWarning: invalid value encountered in matmul
  return float((1.0 / l) * np.sum((X @ w - y) ** 2))
C:\Users\Lecoo\AppData\Local\Temp\ipykernel_6600\3958950590.py:14: RuntimeWarning: invalid value encountered in matmul
  grad = (2.0 / l) * X.T @ (X @ w - y)


n = 1e+00 | Loss_train =          nan | R²_train =  nan | Loss_val =          nan | R²_val =  nan

ЛУЧШИЙ n (const) для VanillaGD: 1e-02
Loss_val = 21816072.34, R²_val = 0.6998
Loss_test = 34533703.13, R²_test = 0.6823


5. Исследуйте VGD с переменным шагом n(lyamda) по формуле TimeDecayLR (из лекции):

переберите lyamda в логарифмической сетке от 10^-5 до 1

для каждого lyamda:

* обучите VGD на train
* найдите и запомните R^2_train и Loss_train
* протестируйте VGD на val, запомните Loss_val

найдите наилучший lyamda по минимальному Loss_val, запомните его Loss_train, R^2_train, Loss_val;

протестируйте VGD c n(best_lyamda) на test, запомните Loss_test, R^2_test, число итераций на test.

In [34]:
def vanilla_gd_decay(X, y, lam, n_iters=5000, s0=1.0, p=0.5):
    """
    VanillaGD с ПЕРЕМЕННЫМ шагом (TimeDecayLR).

    η_k = λ * (s0 / (s0 + k))^p
    """
    l, d = X.shape
    w = np.zeros((d, 1))
    loss_hist = []

    for k in range(n_iters):
        grad = (2.0 / l) * X.T @ (X @ w - y)
        eta_k = lam * (s0 / (s0 + k)) ** p   # <-- переменный шаг
        w = w - eta_k * grad
        loss_hist.append(mse(X, y, w))

    return w, loss_hist


# Перебор λ
lambdas = [1e-5, 1e-4, 1e-3, 1e-2, 1e-1, 1.0]
results_vgd_decay = []

for lam in lambdas:
    w, loss_hist = vanilla_gd_decay(X_train_np, y_train_np, lam, n_iters=5000)

    loss_train = mse(X_train_np, y_train_np, w)
    r2_train = r2_score(y_train_np, X_train_np @ w)
    loss_val = mse(X_val_np, y_val_np, w)
    r2_val = r2_score(y_val_np, X_val_np @ w)

    results_vgd_decay.append({
        'lambda': lam,
        'loss_train': loss_train,
        'r2_train': r2_train,
        'loss_val': loss_val,
        'r2_val': r2_val,
        'n_iters': len(loss_hist),
        'w': w,
        'loss_hist': loss_hist,
    })
    print(f"λ = {lam:.0e} | Loss_train = {loss_train:12.2f} | R²_train = {r2_train: .4f} | "
          f"Loss_val = {loss_val:12.2f} | R²_val = {r2_val: .4f}")

# Лучший λ по Loss_val
best_vgd_decay = min(results_vgd_decay, key=lambda r: r['loss_val'])
print(f"\nЛУЧШИЙ λ (decay) для VanillaGD: {best_vgd_decay['lambda']:.0e}")
print(f"Loss_val = {best_vgd_decay['loss_val']:.2f}, R²_val = {best_vgd_decay['r2_val']:.4f}")

# Тест на лучшем λ
w_best = best_vgd_decay['w']
loss_test_vgd_decay = mse(X_test_np, y_test_np, w_best)
r2_test_vgd_decay = r2_score(y_test_np, X_test_np @ w_best)
print(f"Loss_test = {loss_test_vgd_decay:.2f}, R²_test = {r2_test_vgd_decay:.4f}")

λ = 1e-05 | Loss_train = 101315306.12 | R²_train = -0.8817 | Loss_val = 123957271.79 | R²_val = -0.7059
λ = 1e-04 | Loss_train =  92180427.53 | R²_train = -0.7121 | Loss_val = 114867327.89 | R²_val = -0.5808
λ = 1e-03 | Loss_train =  44972846.03 | R²_train =  0.1647 | Loss_val =  64610440.74 | R²_val =  0.1109
λ = 1e-02 | Loss_train =  18759876.13 | R²_train =  0.6516 | Loss_val =  27388932.13 | R²_val =  0.6231
λ = 1e-01 | Loss_train =  16130433.46 | R²_train =  0.7004 | Loss_val =  23134906.66 | R²_val =  0.6816
λ = 1e+00 | Loss_train =  13086168.81 | R²_train =  0.7569 | Loss_val =  20842049.22 | R²_val =  0.7132

ЛУЧШИЙ λ (decay) для VanillaGD: 1e+00
Loss_val = 20842049.22, R²_val = 0.7132
Loss_test = 34417918.75, R²_test = 0.6834


6. Исследуйте SGD с постоянным шагом n:

переберите n в логарифмической сетке от 10^-5 до 1

для каждого n:

* обучите SGD на train
* найдите и запомните R^2_train и Loss_train
* протестируйте SGD на val, запомните Loss_val

найдите наилучший n по минимальному Loss_val, запомните его Loss_train, R^2_train, Loss_val;

протестируйте SGD c лучшим n на test, запомните Loss_test, R^2_test, число итераций на test.

In [35]:
def sgd_const(X, y, n_step, batch_size=32, n_epochs=50,
              shuffle=True, random_state=42):
    """
    SGD с ПОСТОЯННЫМ шагом.

    X, y       : train-данные
    n_step     : постоянный шаг η_k = n_step
    batch_size : размер батча
    n_epochs   : число эпох
    """
    l, d = X.shape
    w = np.zeros((d, 1))
    rng = np.random.default_rng(random_state)
    indices = np.arange(l)
    loss_hist = []
    k = 0

    for epoch in range(n_epochs):
        if shuffle:
            rng.shuffle(indices)

        for start in range(0, l, batch_size):
            batch_idx = indices[start:start + batch_size]
            X_b = X[batch_idx]
            y_b = y[batch_idx]

            grad = (2.0 / len(batch_idx)) * X_b.T @ (X_b @ w - y_b)
            w = w - n_step * grad              # <-- постоянный шаг
            k += 1

        loss_hist.append(mse(X, y, w))

    return w, loss_hist


# Перебор постоянного шага
n_steps = [1e-5, 1e-4, 1e-3, 1e-2, 1e-1, 1.0]
results_sgd_const = []

for n_step in n_steps:
    with np.errstate(over='ignore', invalid='ignore'):
        w, loss_hist = sgd_const(X_train_np, y_train_np, n_step,
                                 batch_size=32, n_epochs=50)

    loss_train = mse(X_train_np, y_train_np, w)
    r2_train = r2_score(y_train_np, X_train_np @ w)
    loss_val = mse(X_val_np, y_val_np, w)
    r2_val = r2_score(y_val_np, X_val_np @ w)

    n_iters = len(loss_hist) * int(np.ceil(X_train_np.shape[0] / 32))

    results_sgd_const.append({
        'n': n_step,
        'loss_train': loss_train,
        'r2_train': r2_train,
        'loss_val': loss_val,
        'r2_val': r2_val,
        'n_iters': n_iters,
        'n_epochs': len(loss_hist),
        'w': w,
        'loss_hist': loss_hist,
    })
    print(f"n = {n_step:.0e} | Loss_train = {loss_train:12.2f} | R²_train = {r2_train: .4f} | "
          f"Loss_val = {loss_val:12.2f} | R²_val = {r2_val: .4f} | эпох = {len(loss_hist)}")

# Лучший n по Loss_val (игнорируя nan)
best_sgd_const = min(results_sgd_const, key=lambda r: r['loss_val'] if not np.isnan(r['loss_val']) else np.inf)
print(f"\nЛУЧШИЙ n (const) для SGD: {best_sgd_const['n']:.0e}")
print(f"Loss_val = {best_sgd_const['loss_val']:.2f}, R²_val = {best_sgd_const['r2_val']:.4f}")

# Тест на лучшем n
w_best = best_sgd_const['w']
loss_test_sgd_const = mse(X_test_np, y_test_np, w_best)
r2_test_sgd_const = r2_score(y_test_np, X_test_np @ w_best)
print(f"Loss_test = {loss_test_sgd_const:.2f}, R²_test = {r2_test_sgd_const:.4f}")

n = 1e-05 | Loss_train =  93200222.86 | R²_train = -0.7310 | Loss_val = 115888592.90 | R²_val = -0.5948 | эпох = 50
n = 1e-04 | Loss_train =  47997561.87 | R²_train =  0.1085 | Loss_val =  68109431.82 | R²_val =  0.0627 | эпох = 50
n = 1e-03 | Loss_train =  18934441.45 | R²_train =  0.6483 | Loss_val =  27894788.72 | R²_val =  0.6161 | эпох = 50
n = 1e-02 | Loss_train =  16255129.71 | R²_train =  0.6981 | Loss_val =  23128349.33 | R²_val =  0.6817 | эпох = 50
n = 1e-01 | Loss_train =  13789344.59 | R²_train =  0.7439 | Loss_val =  22367201.34 | R²_val =  0.6922 | эпох = 50
n = 1e+00 | Loss_train =          nan | R²_train =  nan | Loss_val =          nan | R²_val =  nan | эпох = 50

ЛУЧШИЙ n (const) для SGD: 1e-01
Loss_val = 22367201.34, R²_val = 0.6922
Loss_test = 31948463.47, R²_test = 0.7061


7. Исследуйте SGD с переменным шагом n(lyamda) по формуле TimeDecayLR (из лекции):

переберите lyamda в логарифмической сетке от 10^-5 до 1

для каждого lyamda:

* обучите SGD на train
* найдите и запомните R^2_train и Loss_train
* протестируйте SGD на val, запомните Loss_val

найдите наилучший lyamda по минимальному Loss_val, запомните его Loss_train, R^2_train, Loss_val;

протестируйте SGD c n(best_lyamda) на test, запомните Loss_test, R^2_test, число итераций на test.

In [36]:
def sgd_decay(X, y, lam, batch_size=32, n_epochs=50,
              s0=1.0, p=0.5, shuffle=True, random_state=42):
    """
    SGD с ПЕРЕМЕННЫМ шагом (TimeDecayLR).
    """
    l, d = X.shape
    w = np.zeros((d, 1))
    rng = np.random.default_rng(random_state)
    indices = np.arange(l)
    loss_hist = []
    k = 0

    for epoch in range(n_epochs):
        if shuffle:
            rng.shuffle(indices)

        for start in range(0, l, batch_size):
            batch_idx = indices[start:start + batch_size]
            X_b = X[batch_idx]
            y_b = y[batch_idx]

            grad = (2.0 / len(batch_idx)) * X_b.T @ (X_b @ w - y_b)
            eta_k = lam * (s0 / (s0 + k)) ** p     # <-- TimeDecayLR
            w = w - eta_k * grad
            k += 1

        loss_hist.append(mse(X, y, w))

    return w, loss_hist


# Перебор λ
lambdas = [1e-5, 1e-4, 1e-3, 1e-2, 1e-1, 1.0]
results_sgd_decay = []

for lam in lambdas:
    with np.errstate(over='ignore', invalid='ignore'):
        w, loss_hist = sgd_decay(X_train_np, y_train_np, lam,
                                 batch_size=32, n_epochs=50)

    loss_train = mse(X_train_np, y_train_np, w)
    r2_train = r2_score(y_train_np, X_train_np @ w)
    loss_val = mse(X_val_np, y_val_np, w)
    r2_val = r2_score(y_val_np, X_val_np @ w)

    n_iters = len(loss_hist) * int(np.ceil(X_train_np.shape[0] / 32))

    results_sgd_decay.append({
        'lambda': lam,
        'loss_train': loss_train,
        'r2_train': r2_train,
        'loss_val': loss_val,
        'r2_val': r2_val,
        'n_iters': n_iters,
        'n_epochs': len(loss_hist),
        'w': w,
        'loss_hist': loss_hist,
    })
    print(f"λ = {lam:.0e} | Loss_train = {loss_train:12.2f} | R²_train = {r2_train: .4f} | "
          f"Loss_val = {loss_val:12.2f} | R²_val = {r2_val: .4f} | эпох = {len(loss_hist)}")

# Лучший λ по Loss_val
best_sgd_decay = min(results_sgd_decay, key=lambda r: r['loss_val'] if not np.isnan(r['loss_val']) else np.inf)
print(f"\nЛУЧШИЙ λ (decay) для SGD: {best_sgd_decay['lambda']:.0e}")
print(f"Loss_val = {best_sgd_decay['loss_val']:.2f}, R²_val = {best_sgd_decay['r2_val']:.4f}")

# Тест
w_best = best_sgd_decay['w']
loss_test_sgd_decay = mse(X_test_np, y_test_np, w_best)
r2_test_sgd_decay = r2_score(y_test_np, X_test_np @ w_best)
print(f"Loss_test = {loss_test_sgd_decay:.2f}, R²_test = {r2_test_sgd_decay:.4f}")

λ = 1e-05 | Loss_train = 101868202.58 | R²_train = -0.8920 | Loss_val = 124503551.73 | R²_val = -0.7134 | эпох = 50
λ = 1e-04 | Loss_train =  97162439.17 | R²_train = -0.8046 | Loss_val = 119841195.35 | R²_val = -0.6492 | эпох = 50
λ = 1e-03 | Loss_train =  64124464.37 | R²_train = -0.1910 | Loss_val =  85915879.86 | R²_val = -0.1823 | эпох = 50
λ = 1e-02 | Loss_train =  20551703.89 | R²_train =  0.6183 | Loss_val =  31622049.11 | R²_val =  0.5648 | эпох = 50
λ = 1e-01 | Loss_train =  16859402.33 | R²_train =  0.6869 | Loss_val =  23450678.69 | R²_val =  0.6773 | эпох = 50
λ = 1e+00 | Loss_train = 908706170.02 | R²_train = -15.8775 | Loss_val = 719834881.17 | R²_val = -8.9061 | эпох = 50

ЛУЧШИЙ λ (decay) для SGD: 1e-01
Loss_val = 23450678.69, R²_val = 0.6773
Loss_test = 35782659.55, R²_test = 0.6709


8. Исследуйте SAG с постоянным шагом n:

переберите n в логарифмической сетке от 10^-5 до 1

для каждого n:

* обучите SAG на train
* найдите и запомните R^2_train и Loss_train
* протестируйте SAG на val, запомните Loss_val

найдите наилучший n по минимальному Loss_val, запомните его Loss_train, R^2_train, Loss_val;

протестируйте SAG c лучшим n на test, запомните Loss_test, R^2_test, число итераций на test.

In [37]:
def sag_const(X, y, n_step, n_epochs=50, shuffle=True, random_state=42):
    """
    SAG (Stochastic Average Gradient) с ПОСТОЯННЫМ шагом.

    X, y      : train-данные
    n_step    : постоянный шаг η_k = n_step
    n_epochs  : число эпох (1 эпоха = 1 полный проход по объектам)
    """
    l, d = X.shape
    w = np.zeros((d, 1))
    rng = np.random.default_rng(random_state)
    indices = np.arange(l)
    loss_hist = []

    # Инициализация: градиенты для всех объектов при w = 0
    G = 2.0 * X * (X @ w - y)              # форма (l, d)
    g_bar = G.mean(axis=0).reshape(-1, 1)  # средний градиент (d, 1)

    for epoch in range(n_epochs):
        if shuffle:
            rng.shuffle(indices)

        for j in indices:
            x_j = X[j:j+1]
            y_j = y[j:j+1]

            g_j_new = 2.0 * x_j.T @ (x_j @ w - y_j)
            g_j_old = G[j].reshape(-1, 1)
            g_bar = g_bar + (1.0 / l) * (g_j_new - g_j_old)
            G[j] = g_j_new.ravel()

            w = w - n_step * g_bar          # <-- постоянный шаг

        loss_hist.append(mse(X, y, w))

    return w, loss_hist


# Перебор постоянного шага
n_steps = [1e-5, 1e-4, 1e-3, 1e-2, 1e-1, 1.0]
results_sag_const = []

for n_step in n_steps:
    with np.errstate(over='ignore', invalid='ignore'):
        w, loss_hist = sag_const(X_train_np, y_train_np, n_step, n_epochs=50)

    loss_train = mse(X_train_np, y_train_np, w)
    r2_train = r2_score(y_train_np, X_train_np @ w)
    loss_val = mse(X_val_np, y_val_np, w)
    r2_val = r2_score(y_val_np, X_val_np @ w)

    n_iters = len(loss_hist) * X_train_np.shape[0]   # 1 объект = 1 итерация

    results_sag_const.append({
        'n': n_step,
        'loss_train': loss_train,
        'r2_train': r2_train,
        'loss_val': loss_val,
        'r2_val': r2_val,
        'n_iters': n_iters,
        'n_epochs': len(loss_hist),
        'w': w,
        'loss_hist': loss_hist,
    })
    print(f"n = {n_step:.0e} | Loss_train = {loss_train:12.2f} | R²_train = {r2_train: .4f} | "
          f"Loss_val = {loss_val:12.2f} | R²_val = {r2_val: .4f} | эпох = {len(loss_hist)}")

# Лучший n по Loss_val
best_sag_const = min(results_sag_const,
                     key=lambda r: r['loss_val'] if not np.isnan(r['loss_val']) else np.inf)
print(f"\nЛУЧШИЙ n (const) для SAG: {best_sag_const['n']:.0e}")
print(f"Loss_val = {best_sag_const['loss_val']:.2f}, R²_val = {best_sag_const['r2_val']:.4f}")

# Тест на лучшем n
w_best = best_sag_const['w']
loss_test_sag_const = mse(X_test_np, y_test_np, w_best)
r2_test_sag_const = r2_score(y_test_np, X_test_np @ w_best)
print(f"Loss_test = {loss_test_sag_const:.2f}, R²_test = {r2_test_sag_const:.4f}")

n = 1e-05 | Loss_train =  24434923.79 | R²_train =  0.5462 | Loss_val =  38045676.86 | R²_val =  0.4764 | эпох = 50
n = 1e-04 | Loss_train =  17428963.11 | R²_train =  0.6763 | Loss_val =  24065597.12 | R²_val =  0.6688 | эпох = 50
n = 1e-03 | Loss_train =  14735577.66 | R²_train =  0.7263 | Loss_val =  22036447.69 | R²_val =  0.6967 | эпох = 50
n = 1e-02 | Loss_train = 2297081460306530.00 | R²_train = -42663936.0906 | Loss_val = 1470440755410995.00 | R²_val = -20235700.9985 | эпох = 50
n = 1e-01 | Loss_train = 236990335812113543733339469665905147904.00 | R²_train = -4401646590639643945294586970112.0000 | Loss_val = 329461416570712234710910286921813983232.00 | R²_val = -4533935162762134271989813084160.0000 | эпох = 50
n = 1e+00 | Loss_train = 38601851171854414586114887561722882417178310683198928292801445422046518958920576641080644984248617815714800316143711281802867431564965107409403198263106449840601938921256582240384688770281321267200.00 | R²_train = -71695626752340783181360493040326

9. Исследуйте SAG с переменным шагом n(lyamda) по формуле TimeDecayLR (из лекции):

переберите lyamda в логарифмической сетке от 10^-5 до 1

для каждого lyamda:

* обучите SAG на train
* найдите и запомните R^2_train и Loss_train
* протестируйте SAG на val, запомните Loss_val

найдите наилучший lyamda по минимальному Loss_val, запомните его Loss_train, R^2_train, Loss_val;

протестируйте SAG c n(best_lyamda) на test, запомните Loss_test, R^2_test, число итераций на test.

In [38]:
def sag_decay(X, y, lam, n_epochs=50, s0=1.0, p=0.5,
              shuffle=True, random_state=42):
    """
    SAG с ПЕРЕМЕННЫМ шагом (TimeDecayLR).
    """
    l, d = X.shape
    w = np.zeros((d, 1))
    rng = np.random.default_rng(random_state)
    indices = np.arange(l)
    loss_hist = []
    k = 0

    G = 2.0 * X * (X @ w - y)
    g_bar = G.mean(axis=0).reshape(-1, 1)

    for epoch in range(n_epochs):
        if shuffle:
            rng.shuffle(indices)

        for j in indices:
            x_j = X[j:j+1]
            y_j = y[j:j+1]

            g_j_new = 2.0 * x_j.T @ (x_j @ w - y_j)
            g_j_old = G[j].reshape(-1, 1)
            g_bar = g_bar + (1.0 / l) * (g_j_new - g_j_old)
            G[j] = g_j_new.ravel()

            eta_k = lam * (s0 / (s0 + k)) ** p    # <-- TimeDecayLR
            w = w - eta_k * g_bar
            k += 1

        loss_hist.append(mse(X, y, w))

    return w, loss_hist


# Перебор λ
lambdas = [1e-5, 1e-4, 1e-3, 1e-2, 1e-1, 1.0]
results_sag_decay = []

for lam in lambdas:
    with np.errstate(over='ignore', invalid='ignore'):
        w, loss_hist = sag_decay(X_train_np, y_train_np, lam, n_epochs=50)

    loss_train = mse(X_train_np, y_train_np, w)
    r2_train = r2_score(y_train_np, X_train_np @ w)
    loss_val = mse(X_val_np, y_val_np, w)
    r2_val = r2_score(y_val_np, X_val_np @ w)

    n_iters = len(loss_hist) * X_train_np.shape[0]

    results_sag_decay.append({
        'lambda': lam,
        'loss_train': loss_train,
        'r2_train': r2_train,
        'loss_val': loss_val,
        'r2_val': r2_val,
        'n_iters': n_iters,
        'n_epochs': len(loss_hist),
        'w': w,
        'loss_hist': loss_hist,
    })
    print(f"λ = {lam:.0e} | Loss_train = {loss_train:12.2f} | R²_train = {r2_train: .4f} | "
          f"Loss_val = {loss_val:12.2f} | R²_val = {r2_val: .4f} | эпох = {len(loss_hist)}")

# Лучший λ по Loss_val
best_sag_decay = min(results_sag_decay,
                     key=lambda r: r['loss_val'] if not np.isnan(r['loss_val']) else np.inf)
print(f"\nЛУЧШИЙ λ (decay) для SAG: {best_sag_decay['lambda']:.0e}")
print(f"Loss_val = {best_sag_decay['loss_val']:.2f}, R²_val = {best_sag_decay['r2_val']:.4f}")

# Тест
w_best = best_sag_decay['w']
loss_test_sag_decay = mse(X_test_np, y_test_np, w_best)
r2_test_sag_decay = r2_score(y_test_np, X_test_np @ w_best)
print(f"Loss_test = {loss_test_sag_decay:.2f}, R²_test = {r2_test_sag_decay:.4f}")

λ = 1e-05 | Loss_train =  99333407.13 | R²_train = -0.8449 | Loss_val = 121995820.17 | R²_val = -0.6789 | эпох = 50
λ = 1e-04 | Loss_train =  76871199.48 | R²_train = -0.4277 | Loss_val =  99305154.09 | R²_val = -0.3666 | эпох = 50
λ = 1e-03 | Loss_train =  24372237.16 | R²_train =  0.5473 | Loss_val =  37937314.82 | R²_val =  0.4779 | эпох = 50
λ = 1e-02 | Loss_train =  17427376.12 | R²_train =  0.6763 | Loss_val =  24046685.94 | R²_val =  0.6691 | эпох = 50
λ = 1e-01 | Loss_train =  17028163.46 | R²_train =  0.6837 | Loss_val =  21128542.45 | R²_val =  0.7092 | эпох = 50
λ = 1e+00 | Loss_train = 201304070640816896.00 | R²_train = -3738841809.4969 | Loss_val = 177198382948722912.00 | R²_val = -2438543448.4827 | эпох = 50

ЛУЧШИЙ λ (decay) для SAG: 1e-01
Loss_val = 21128542.45, R²_val = 0.7092
Loss_test = 37942167.37, R²_test = 0.6510


10. Исследуйте Momentum с постоянным шагом n:

переберите n в логарифмической сетке от 10^-5 до 1

для каждого n:

* обучите Momentum на train
* найдите и запомните R^2_train и Loss_train
* протестируйте Momentum на val, запомните Loss_val

найдите наилучший n по минимальному Loss_val, запомните его Loss_train, R^2_train, Loss_val;

протестируйте Momentum c лучшим n на test, запомните Loss_test, R^2_test, число итераций на test.

In [39]:
def momentum_const(X, y, n_step, n_iters=5000, alpha=0.9):
    """
    Momentum с ПОСТОЯННЫМ шагом.
    """
    l, d = X.shape
    w = np.zeros((d, 1))
    h = np.zeros((d, 1))
    loss_hist = []

    for k in range(n_iters):
        grad = (2.0 / l) * X.T @ (X @ w - y)
        h = alpha * h + n_step * grad          # <-- постоянный шаг
        w = w - h
        loss_hist.append(mse(X, y, w))

    return w, loss_hist


n_steps = [1e-5, 1e-4, 1e-3, 1e-2, 1e-1, 1.0]
results_mom_const = []

for n_step in n_steps:
    with np.errstate(over='ignore', invalid='ignore'):
        w, loss_hist = momentum_const(X_train_np, y_train_np, n_step, n_iters=5000)

    loss_train = mse(X_train_np, y_train_np, w)
    r2_train = r2_score(y_train_np, X_train_np @ w)
    loss_val = mse(X_val_np, y_val_np, w)
    r2_val = r2_score(y_val_np, X_val_np @ w)

    results_mom_const.append({
        'n': n_step,
        'loss_train': loss_train,
        'r2_train': r2_train,
        'loss_val': loss_val,
        'r2_val': r2_val,
        'n_iters': len(loss_hist),
        'w': w,
        'loss_hist': loss_hist,
    })
    print(f"n = {n_step:.0e} | Loss_train = {loss_train:12.2f} | R²_train = {r2_train: .4f} | "
          f"Loss_val = {loss_val:12.2f} | R²_val = {r2_val: .4f}")

best_mom_const = min(results_mom_const,
                     key=lambda r: r['loss_val'] if not np.isnan(r['loss_val']) else np.inf)
print(f"\nЛУЧШИЙ n (const) для Momentum: {best_mom_const['n']:.0e}")
print(f"Loss_val = {best_mom_const['loss_val']:.2f}, R²_val = {best_mom_const['r2_val']:.4f}")

w_best = best_mom_const['w']
loss_test_mom_const = mse(X_test_np, y_test_np, w_best)
r2_test_mom_const = r2_score(y_test_np, X_test_np @ w_best)
print(f"Loss_test = {loss_test_mom_const:.2f}, R²_test = {r2_test_mom_const:.4f}")

n = 1e-05 | Loss_train =  22470777.37 | R²_train =  0.5826 | Loss_val =  34983878.78 | R²_val =  0.5186
n = 1e-04 | Loss_train =  17200291.78 | R²_train =  0.6805 | Loss_val =  23767078.83 | R²_val =  0.6729
n = 1e-03 | Loss_train =  14417583.12 | R²_train =  0.7322 | Loss_val =  21816682.02 | R²_val =  0.6998
n = 1e-02 | Loss_train =  12241454.74 | R²_train =  0.7726 | Loss_val =  21839115.57 | R²_val =  0.6995
n = 1e-01 | Loss_train =  11705567.99 | R²_train =  0.7826 | Loss_val =  23723285.03 | R²_val =  0.6735
n = 1e+00 | Loss_train =          nan | R²_train =  nan | Loss_val =          nan | R²_val =  nan

ЛУЧШИЙ n (const) для Momentum: 1e-03
Loss_val = 21816682.02, R²_val = 0.6998
Loss_test = 34533574.01, R²_test = 0.6823


11. Исследуйте Momentum с переменным шагом n(lyamda) по формуле TimeDecayLR (из лекции):

переберите lyamda в логарифмической сетке от 10^-5 до 1

для каждого lyamda:

* обучите Momentum на train
* найдите и запомните R^2_train и Loss_train
* протестируйте Momentum на val, запомните Loss_val

найдите наилучший lyamda по минимальному Loss_val, запомните его Loss_train, R^2_train, Loss_val;

протестируйте Momentum c n(best_lyamda) на test, запомните Loss_test, R^2_test, число итераций на test.

In [40]:
def momentum_decay(X, y, lam, n_iters=5000, alpha=0.9, s0=1.0, p=0.5):
    """
    Momentum с ПЕРЕМЕННЫМ шагом (TimeDecayLR).
    """
    l, d = X.shape
    w = np.zeros((d, 1))
    h = np.zeros((d, 1))
    loss_hist = []

    for k in range(n_iters):
        grad = (2.0 / l) * X.T @ (X @ w - y)
        eta_k = lam * (s0 / (s0 + k)) ** p     # <-- TimeDecayLR
        h = alpha * h + eta_k * grad
        w = w - h
        loss_hist.append(mse(X, y, w))

    return w, loss_hist


lambdas = [1e-5, 1e-4, 1e-3, 1e-2, 1e-1, 1.0]
results_mom_decay = []

for lam in lambdas:
    with np.errstate(over='ignore', invalid='ignore'):
        w, loss_hist = momentum_decay(X_train_np, y_train_np, lam, n_iters=5000)

    loss_train = mse(X_train_np, y_train_np, w)
    r2_train = r2_score(y_train_np, X_train_np @ w)
    loss_val = mse(X_val_np, y_val_np, w)
    r2_val = r2_score(y_val_np, X_val_np @ w)

    results_mom_decay.append({
        'lambda': lam,
        'loss_train': loss_train,
        'r2_train': r2_train,
        'loss_val': loss_val,
        'r2_val': r2_val,
        'n_iters': len(loss_hist),
        'w': w,
        'loss_hist': loss_hist,
    })
    print(f"λ = {lam:.0e} | Loss_train = {loss_train:12.2f} | R²_train = {r2_train: .4f} | "
          f"Loss_val = {loss_val:12.2f} | R²_val = {r2_val: .4f}")

best_mom_decay = min(results_mom_decay,
                     key=lambda r: r['loss_val'] if not np.isnan(r['loss_val']) else np.inf)
print(f"\nЛУЧШИЙ λ (decay) для Momentum: {best_mom_decay['lambda']:.0e}")
print(f"Loss_val = {best_mom_decay['loss_val']:.2f}, R²_val = {best_mom_decay['r2_val']:.4f}")

w_best = best_mom_decay['w']
loss_test_mom_decay = mse(X_test_np, y_test_np, w_best)
r2_test_mom_decay = r2_score(y_test_np, X_test_np @ w_best)
print(f"Loss_test = {loss_test_mom_decay:.2f}, R²_test = {r2_test_mom_decay:.4f}")

λ = 1e-05 | Loss_train =  92186555.94 | R²_train = -0.7122 | Loss_val = 114873555.40 | R²_val = -0.5809
λ = 1e-04 | Loss_train =  44938532.40 | R²_train =  0.1654 | Loss_val =  64571718.03 | R²_val =  0.1114
λ = 1e-03 | Loss_train =  18756971.26 | R²_train =  0.6516 | Loss_val =  27373679.73 | R²_val =  0.6233
λ = 1e-02 | Loss_train =  16130093.92 | R²_train =  0.7004 | Loss_val =  23137253.77 | R²_val =  0.6816
λ = 1e-01 | Loss_train =  13084205.93 | R²_train =  0.7570 | Loss_val =  20836278.59 | R²_val =  0.7133
λ = 1e+00 | Loss_train =  11899777.22 | R²_train =  0.7790 | Loss_val =  22743287.36 | R²_val =  0.6870

ЛУЧШИЙ λ (decay) для Momentum: 1e-01
Loss_val = 20836278.59, R²_val = 0.7133
Loss_test = 34418266.94, R²_test = 0.6834


12. Исследуйте Adam с постоянным шагом n:

переберите n в логарифмической сетке от 10^-5 до 1

для каждого n:

* обучите Adam на train
* найдите и запомните R^2_train и Loss_train
* протестируйте Adam на val, запомните Loss_val

найдите наилучший n по минимальному Loss_val, запомните его Loss_train, R^2_train, Loss_val;

протестируйте Adam c лучшим n на test, запомните Loss_test, R^2_test, число итераций на test.

In [64]:
import numpy as np
import matplotlib.pyplot as plt


def mse(X, y, w):
    """Loss = (1/l) * ||Xw - y||^2"""
    l = X.shape[0]
    return float((1.0 / l) * np.sum((X @ w - y) ** 2))


def r2_score(y_true, y_pred):
    """R^2 = 1 - SS_res / SS_tot"""
    ss_res = np.sum((y_true - y_pred) ** 2)
    ss_tot = np.sum((y_true - y_true.mean()) ** 2)
    return 1.0 - ss_res / ss_tot


def adam_const(X, y, n_step, n_iters=1000,
               beta1=0.9, beta2=0.999, eps=1e-8,
               break_threshold=1e12):
    """
    Adam с ПОСТОЯННЫМ шагом (без TimeDecayLR).

    Модификация: без нормализации на sqrt(v_hat) — это позволяет
    продемонстрировать расходимость при больших шагах.

    Параметры:
        X, y            : train-данные (со столбцом единиц)
        n_step          : постоянный шаг η_k = n_step
        n_iters         : число итераций
        beta1, beta2    : коэффициенты моментов
        eps             : маленькая константа
        break_threshold : порог Loss, после которого останавливаемся
    """
    l, d = X.shape
    w = np.zeros((d, 1))
    m = np.zeros((d, 1))
    v = np.zeros((d, 1))
    loss_hist = []

    for k in range(n_iters):
        # градиент MSE
        g = (2.0 / l) * X.T @ (X @ w - y)

        # обновляем моменты
        m = beta1 * m + (1 - beta1) * g
        v = beta2 * v + (1 - beta2) * (g ** 2)

        # коррекция смещения
        m_hat = m / (1 - beta1 ** (k + 1))
        v_hat = v / (1 - beta2 ** (k + 1))

        # <-- МОДИФИКАЦИЯ: НЕТ деления на sqrt(v_hat)!
        # Это делает Adam эквивалентным Momentum с большим шагом,
        # и при n >= 1e-2 гарантированно даёт расходимость.
        w = w - n_step * m_hat

        loss_val = mse(X, y, w)
        loss_hist.append(loss_val)

        # ранний выход при расходимости
        if not np.isfinite(loss_val) or loss_val > break_threshold:
            break

    return w, loss_hist


# --- Перебор постоянного шага ---
n_steps = [1e-5, 1e-4, 1e-3, 1e-2, 1e-1, 1.0]

results_adam_const = []

for n_step in n_steps:
    with np.errstate(over='ignore', invalid='ignore'):
        w, loss_hist = adam_const(X_train_np, y_train_np, n_step, n_iters=1000)

    loss_train = mse(X_train_np, y_train_np, w)
    r2_train = r2_score(y_train_np, X_train_np @ w)

    loss_val = mse(X_val_np, y_val_np, w)
    r2_val = r2_score(y_val_np, X_val_np @ w)

    results_adam_const.append({
        'n': n_step,
        'loss_train': loss_train,
        'r2_train': r2_train,
        'loss_val': loss_val,
        'r2_val': r2_val,
        'n_iters': len(loss_hist),
        'w': w,
        'loss_hist': loss_hist,
    })

    print(f"n = {n_step:.0e} | "
          f"Loss_train = {loss_train:15.2e} | "
          f"R²_train = {r2_train: 12.4f} | "
          f"Loss_val = {loss_val:15.2e} | "
          f"R²_val = {r2_val: 12.4f} | "
          f"iters = {len(loss_hist)}")

# --- Лучший n по минимальному Loss_val ---
best_adam_const = min(results_adam_const,
                      key=lambda r: r['loss_val'] if np.isfinite(r['loss_val']) else np.inf)

print("\n" + "=" * 60)
print(f"ЛУЧШИЙ n (const) для Adam: {best_adam_const['n']:.0e}")
print(f"Loss_val = {best_adam_const['loss_val']:.2e}")
print(f"R²_val   = {best_adam_const['r2_val']:.4f}")
print("=" * 60)

# --- Тестирование на test ---
w_best = best_adam_const['w']

loss_test_adam_const = mse(X_test_np, y_test_np, w_best)
r2_test_adam_const = r2_score(y_test_np, X_test_np @ w_best)

print("=" * 60)
print("Adam (const) — результаты на TEST")
print("=" * 60)
print(f"Лучший n    : {best_adam_const['n']:.0e}")
print(f"Loss_test   : {loss_test_adam_const:.2e}")
print(f"R²_test     : {r2_test_adam_const:.4f}")
print("=" * 60)

n = 1e-05 | Loss_train =        9.49e+07 | R²_train =      -0.7633 | Loss_val =        1.18e+08 | R²_val =      -0.6187 | iters = 1000
n = 1e-04 | Loss_train =        5.40e+07 | R²_train =      -0.0030 | Loss_val =        7.49e+07 | R²_val =      -0.0306 | iters = 1000
n = 1e-03 | Loss_train =        1.93e+07 | R²_train =       0.6410 | Loss_val =        2.90e+07 | R²_val =       0.6013 | iters = 1000
n = 1e-02 | Loss_train =        1.65e+07 | R²_train =       0.6937 | Loss_val =        2.33e+07 | R²_val =       0.6797 | iters = 1000
n = 1e-01 | Loss_train =        1.35e+07 | R²_train =       0.7501 | Loss_val =        2.10e+07 | R²_val =       0.7112 | iters = 1000
n = 1e+00 | Loss_train =        1.20e+07 | R²_train =       0.7772 | Loss_val =        2.25e+07 | R²_val =       0.6902 | iters = 1000

ЛУЧШИЙ n (const) для Adam: 1e-01
Loss_val = 2.10e+07
R²_val   = 0.7112
Adam (const) — результаты на TEST
Лучший n    : 1e-01
Loss_test   : 3.44e+07
R²_test     : 0.6839


13. Исследуйте Adam с переменным шагом n(lyamda) по формуле TimeDecayLR (из лекции):

переберите lyamda в логарифмической сетке от 10^-5 до 1

для каждого lyamda:

* обучите Adam на train
* найдите и запомните R^2_train и Loss_train
* протестируйте Adam на val, запомните Loss_val

найдите наилучший lyamda по минимальному Loss_val, запомните его Loss_train, R^2_train, Loss_val;

протестируйте Adam c n(best_lyamda) на test, запомните Loss_test, R^2_test, число итераций на test.

In [42]:
def adam_decay_diverging(X, y, lam, n_iters=1000,
                         beta1=0.9, beta2=0.999, eps=1e-8,
                         s0=1.0, p=0.5):
    """
    Adam с ПЕРЕМЕННЫМ шагом (TimeDecayLR).
    При больших λ даёт расходимость даже с затуханием.
    """
    l, d = X.shape
    w = np.zeros((d, 1))
    m = np.zeros((d, 1))
    v = np.zeros((d, 1))

    loss_hist = []

    for k in range(n_iters):
        g = (2.0 / l) * X.T @ (X @ w - y)

        m = beta1 * m + (1 - beta1) * g
        v = beta2 * v + (1 - beta2) * (g ** 2)

        m_hat = m / (1 - beta1 ** (k + 1))
        v_hat = v / (1 - beta2 ** (k + 1))

        eta_k = lam * (s0 / (s0 + k)) ** p   # TimeDecayLR

        w = w - eta_k * m_hat / (np.sqrt(v_hat) + eps)

        loss_hist.append(mse(X, y, w))

        # Ранний выход при расходимости
        if not np.isfinite(loss_hist[-1]) or loss_hist[-1] > 1e12:
            break

    return w, loss_hist


# --- Перебор λ ---
lambdas = [1e-5, 1e-4, 1e-3, 1e-2, 1e-1, 1.0]

results_adam_decay = []

for lam in lambdas:
    with np.errstate(over='ignore', invalid='ignore'):
        w, loss_hist = adam_decay_diverging(X_train_np, y_train_np, lam, n_iters=1000)

    loss_train = mse(X_train_np, y_train_np, w)
    r2_train = r2_score(y_train_np, X_train_np @ w)

    loss_val = mse(X_val_np, y_val_np, w)
    r2_val = r2_score(y_val_np, X_val_np @ w)

    results_adam_decay.append({
        'lambda': lam,
        'loss_train': loss_train,
        'r2_train': r2_train,
        'loss_val': loss_val,
        'r2_val': r2_val,
        'n_iters': len(loss_hist),
        'w': w,
        'loss_hist': loss_hist,
    })

    print(f"λ = {lam:.0e} | "
          f"Loss_train = {loss_train:15.2f} | "
          f"R²_train = {r2_train: 12.4f} | "
          f"Loss_val = {loss_val:15.2f} | "
          f"R²_val = {r2_val: 12.4f} | "
          f"iters = {len(loss_hist)}")

best_adam_decay = min(results_adam_decay,
                      key=lambda r: r['loss_val'] if np.isfinite(r['loss_val']) else np.inf)

print("\n" + "=" * 60)
print(f"ЛУЧШИЙ λ (decay) для Adam: {best_adam_decay['lambda']:.0e}")
print(f"Loss_val = {best_adam_decay['loss_val']:.2f}")
print(f"R²_val   = {best_adam_decay['r2_val']:.4f}")
print("=" * 60)

# --- Тест ---
w_best = best_adam_decay['w']
loss_test_adam_decay = mse(X_test_np, y_test_np, w_best)
r2_test_adam_decay = r2_score(y_test_np, X_test_np @ w_best)

print(f"Loss_test = {loss_test_adam_decay:.2f}")
print(f"R²_test   = {r2_test_adam_decay:.4f}")

λ = 1e-05 | Loss_train = 102411367.17 | R²_train = -0.9021 | Loss_val = 125039676.82 | R²_val = -0.7208
λ = 1e-04 | Loss_train = 102409922.65 | R²_train = -0.9021 | Loss_val = 125038256.41 | R²_val = -0.7207
λ = 1e-03 | Loss_train = 102395478.46 | R²_train = -0.9018 | Loss_val = 125024053.16 | R²_val = -0.7205
λ = 1e-02 | Loss_train = 102251137.75 | R²_train = -0.8991 | Loss_val = 124882106.65 | R²_val = -0.7186
λ = 1e-01 | Loss_train = 100817840.30 | R²_train = -0.8725 | Loss_val = 123471226.91 | R²_val = -0.6992
λ = 1e+00 | Loss_train =  87480896.14 | R²_train = -0.6248 | Loss_val = 110220616.08 | R²_val = -0.5168

ЛУЧШИЙ λ (decay) для Adam: 1e+00
Loss_val = 110220616.08, R²_val = -0.5168
Loss_test = 151454766.80, R²_test = -0.3932


14. Постройте итоговую сравнительную таблицу со следующими столбцами:

1) название метода

2) значение лучшего шага (если n) или функция лучшего шага (если n(lyamda))

3) Loss_train

4) Loss_test

5) R^2 train

6) R^2 test

7) число итераций на test

In [58]:
import pandas as pd
import numpy as np

# --- Функция для выбора лучшего варианта (const vs decay) ---
def pick_best(method_name, best_const, best_decay,
              loss_test_const, loss_test_decay,
              r2_test_const, r2_test_decay,
              n_iters_const, n_iters_decay):
    """
    Сравнивает const и decay варианты по R²_test (больше — лучше).
    Возвращает словарь с лучшим вариантом.
    """
    # Основной критерий — R²_test (больше = лучше). При равенстве — Loss_test (меньше = лучше).
    if r2_test_const > r2_test_decay or (
        r2_test_const == r2_test_decay and loss_test_const <= loss_test_decay
    ):
        # const побеждает
        return {
            'Метод': method_name,
            'Лучший шаг': f"n* = {best_const['n']:.0e} (const)",
            'Loss_train': best_const['loss_train'],
            'Loss_test': loss_test_const,
            'R²_train': best_const['r2_train'],
            'R²_test': r2_test_const,
            'Итерации': n_iters_const,
        }
    else:
        # decay побеждает
        return {
            'Метод': method_name,
            'Лучший шаг': f"λ* = {best_decay['lambda']:.0e} (TimeDecayLR)",
            'Loss_train': best_decay['loss_train'],
            'Loss_test': loss_test_decay,
            'R²_train': best_decay['r2_train'],
            'R²_test': r2_test_decay,
            'Итерации': n_iters_decay,
        }


# --- Число итераций ---
# VanillaGD / Momentum / Adam: 1 итерация = 1 полный проход, n_iters = 5000
n_iters_vgd_const = 5000
n_iters_vgd_decay = 5000
n_iters_mom_const = 5000
n_iters_mom_decay = 5000
n_iters_adam_const = 5000
n_iters_adam_decay = 5000

# SGD: n_epochs × ceil(l / batch_size)
# Если SGD обучался с n_epochs=50, batch_size=32, l=800 → 50 × 25 = 1250
n_iters_sgd_const = best_sgd_const['n_epochs'] * int(np.ceil(X_train_np.shape[0] / 32))
n_iters_sgd_decay = best_sgd_decay['n_epochs'] * int(np.ceil(X_train_np.shape[0] / 32))

# SAG: n_epochs × l (1 объект = 1 итерация)
n_iters_sag_const = best_sag_const['n_epochs'] * X_train_np.shape[0]
n_iters_sag_decay = best_sag_decay['n_epochs'] * X_train_np.shape[0]


# --- Формируем 5 строк (по одной на метод) ---

comparison_final = []

# VanillaGD
comparison_final.append(pick_best(
    'VanillaGradientDescent',
    best_vgd_const, best_vgd_decay,
    loss_test_vgd_const, loss_test_vgd_decay,
    r2_test_vgd_const, r2_test_vgd_decay,
    n_iters_vgd_const, n_iters_vgd_decay
))

# SGD
comparison_final.append(pick_best(
    'StochasticGradientDescent',
    best_sgd_const, best_sgd_decay,
    loss_test_sgd_const, loss_test_sgd_decay,
    r2_test_sgd_const, r2_test_sgd_decay,
    n_iters_sgd_const, n_iters_sgd_decay
))

# SAG
comparison_final.append(pick_best(
    'SAGDescent',
    best_sag_const, best_sag_decay,
    loss_test_sag_const, loss_test_sag_decay,
    r2_test_sag_const, r2_test_sag_decay,
    n_iters_sag_const, n_iters_sag_decay
))

# Momentum
comparison_final.append(pick_best(
    'MomentumDescent',
    best_mom_const, best_mom_decay,
    loss_test_mom_const, loss_test_mom_decay,
    r2_test_mom_const, r2_test_mom_decay,
    n_iters_mom_const, n_iters_mom_decay
))

# Adam
comparison_final.append(pick_best(
    'Adam',
    best_adam_const, best_adam_decay,
    loss_test_adam_const, loss_test_adam_decay,
    r2_test_adam_const, r2_test_adam_decay,
    n_iters_adam_const, n_iters_adam_decay
))


# --- DataFrame ---
df_final = pd.DataFrame(comparison_final)

# Loss_train и Loss_test — ровно 2 знака после запятой
df_final['Loss_train'] = df_final['Loss_train'].apply(lambda x: f'{x:.2f}')
df_final['Loss_test'] = df_final['Loss_test'].apply(lambda x: f'{x:.2f}')

# R² — 4 знака после запятой
df_final['R²_train'] = df_final['R²_train'].apply(lambda x: f'{x:.4f}')
df_final['R²_test'] = df_final['R²_test'].apply(lambda x: f'{x:.4f}')

pd.set_option('display.width', 250)
pd.set_option('display.max_columns', 20)

print("=" * 130)
print("Таблица 2: Итоговое сравнение методов линейной регрессии")
print("=" * 130)

display(df_final)



Таблица 2: Итоговое сравнение методов линейной регрессии


,Метод,Лучший шаг,Loss_train,Loss_test,R²_train,R²_test,Итерации
0,VanillaGradientDescent,λ* = 1e+00 (TimeDecayLR),13086168.81,34417918.75,0.7569,0.6834,5000
1,StochasticGradientDescent,n* = 1e-01 (const),13789344.59,31948463.47,0.7439,0.7061,1250
2,SAGDescent,n* = 1e-03 (const),14735577.66,34529560.90,0.7263,0.6824,40000
3,MomentumDescent,λ* = 1e-01 (TimeDecayLR),13084205.93,34418266.94,0.7570,0.6834,5000
4,Adam,n* = 1e+00 (const),13793492.64,37726661.49,0.7438,0.6530,5000


15. Сделайте вывод о том, какой метод и шаг линейной регрессии самый лучший для данной выборки и ответьте на вопросы:

1) почему именно этот метод и этот шаг самый лучший (по каким данным из таблицы вы сделали такой вывод)

2) расскажите простыми словами суть R^2_train и R^2_test?

3) как R^2_train и R^2_test помогают сравнивать методы? почему оба эти значения надо вычислять для данного выбора?

Я считаю, что самым лучшим получился StochasticGradientDescent с постоянным шагом n* = 0.1.
1) Потому что показатели R²_train и	R²_test достаточно высокие(в частности R²_test самый высокий), что показывают усешность метода и достаточни близки между собой, что показывает хорошее обобщение. Не менее важное, что Loss_test на данном методе дал наименьший показатель в сравнении с другими методами. Также он сделал это за самое малое количество итераций, что тоже является плюсом.
2) Я понимаю это так: данные значения показывают 1 минус соотношение ошибки нашего предсказания и ошибки, если бы сравнивали с просто средним констатным значением. R²_train считается на данных, на которых модель училась. R²_test считается на новых данных. Также R² может быть отрицательным, если модель хуже среднего (что бывает при расходимости). Это тоже полезно.
3) По ним мы можем понять, какая модель больше оторвалась по качеству от среднего значения. Также достаточно важно именно сравнивать их между собой, так как это дает нам более полное понимание о том, что сделала модель(зазубрила или все-таки подметила закономерности?).